In [ ]:
import os
import sys
import glob
from pyhere import here
from pathlib import Path

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import seaborn as sns
import torch
import skmisc
import subprocess
from joblib import parallel_backend
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.backends.backend_pdf import PdfPages
import warnings

import zstandard as zstd
import hdf5plugin

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

## Set up parameters

In [ ]:
# Directories
ma.create_directories(dir_path = str(here('data/integrate/first_pass')))
ma.create_directories(dir_path = str(here('data/integrate/first_pass/tmp')))
ma.create_directories(dir_path = str(here('data/integrate/first_pass/embedding')))
ma.create_directories(dir_path = str(here('data/integrate/first_pass/files')))
ma.create_directories(dir_path = str(here('data/integrate/first_pass/plot')))
ma.create_directories(dir_path = str(here('data/integrate/first_pass/objects')))
ma.create_directories(dir_path = str(here('data/integrate/first_pass/models')))

In [ ]:
# Saving
base_dir = str(here('data/integrate/first_pass/'))
file_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
tmp_dir = os.path.join(base_dir, 'tmp') 
emb_dir = os.path.join(base_dir, 'embedding') 
objects_dir = os.path.join(base_dir, 'objects') 

In [ ]:
# Path to enviroments
py_analysis = '/work/islet_cartography_scrna/scrna_cartography_py_analysis'
scpoli = '/work/islet_cartography_scrna/scrna_cartography_scpoli'

In [ ]:
# hvgs 
hvgs = [500, 1000, 2000]

# directories
adata_dir = tmp_dir  # directory with preprocessed HVG files
file_dir = os.path.join(base_dir, 'files')  # directory to save embeddings
model_dir = os.path.join(base_dir, 'models')   # directory to save models

# With donor
key_batch_with_donor = ["technical", "ic_id_donor_overall"]

# Without donor
key_batch_no_donor = ["technical"]

latent_dims = 20
embedding_dims = 20

celltype = 'study_cell_annotation_harmonized' 
celltype_ignore = 'unknown'

## Load data

In [ ]:
adata =sc.read_h5ad(here('data/anndata/AB_combined.h5ad'))

## Add technical covariate

In [ ]:
# Add technical variation
adata.obs['technical'] = (
    adata.obs['library_prep'].astype(str) + "_" + adata.obs['ic_id_study_overall'].astype(str) + "_" + adata.obs['cell_nuclei'].astype(str)
)

In [ ]:
# copy of raw counts
adata.raw = adata.copy()

In [ ]:
for n in hvgs:
    
    ad = adata.copy()

    ########## HVG SELECTION ################
    #Selcet hvgs, use counts layer as these contain the non-normalized values (only read-based methods are normalized to gene length)
    print(f'Finding {n} highly variable genes')
    sc.pp.highly_variable_genes(
        ad,
        n_top_genes=n,
        flavor="seurat_v3",
        layer="counts",
        batch_key="technical",
        subset=True
    )

    # Save HVGs
    hvg_list = ad.var[ad.var['highly_variable']].index.tolist()
    hvg_path = os.path.join(file_dir, f'hvgs_{n}.txt')
    with open(hvg_path, 'w') as f:
        for gene in hvg_list:
            f.write(gene + '\n')
    print(f'Saved HVGs to {hvg_path}')

    # Save adata
    adata_file = os.path.join(objects_dir, f"adata_{n}_hvg.h5ad")
    ad.write(adata_file)
    print(f'Saved adata object to {adata_file}')

    # Save adata for scPoli
    adata_file_scpoli = os.path.join(objects_dir, f"adata_scpoli_{n}_hvg.h5ad")
    del ad.uns["log1p"]
    ad.write(adata_file_scpoli)
    print(f'Saved adata object for scPoli to {adata_file_scpoli}')

    #### NO integrate #########
    key_save_no_int = f"pca_{n}_hvg"
    
    print(f'Running unintegrated PCA: {key_save_no_int}')
    result_int = subprocess.run([
        "conda", "run", "-p", py_analysis, "python", "run_pca.py",
        str(n), adata_file, model_dir, emb_dir, key_save_no_int, str(latent_dims)
    ], capture_output=True, text=True)
    print(result_int.stderr)

    #### integrate ##########
    for key_batch in [key_batch_with_donor, key_batch_no_donor]:
        key_save = f"{'_'.join(key_batch)}_{latent_dims}_{embedding_dims}"
        print(f'Running {key_save}')

        ##### SCPOLI #########
        print('Running scpoli')
        result_scpoli = subprocess.run([
            "conda", "run", "-p", scpoli, "python", "run_scpoli.py",
            str(n), adata_file_scpoli, model_dir, emb_dir,
            ",".join(key_batch), key_save, str(latent_dims), str(embedding_dims)], 
                                       capture_output=True, text=True)
        print(result_scpoli.stderr)

        ##### SCVI ##########
        print('Running scvi')
        if len(key_batch) == 2:  # with donor
            batch_key = key_batch[1]       # donor
            categorical_covariates = [key_batch[0]]  # system
        else:  # without donor
            batch_key = key_batch[0]       # system
            categorical_covariates = None  # no donor
        
        result_scvi = subprocess.run([
            "conda", "run", "-p", py_analysis, "python", "run_scvi.py",
            str(n), adata_file, model_dir, emb_dir,
            batch_key, ",".join(categorical_covariates) if categorical_covariates else "None",
            key_save, str(latent_dims), str(embedding_dims)], 
                                     capture_output=True, text=True)
        print(result_scvi.stderr)

        ##### SYSVI ###########
        print('Running sysvi')
        if len(key_batch) == 2:  # with donor
            batch_key = key_batch[0]       # system
            categorical_covariates = [key_batch[1]]  # donor
        else:  # without donor
            batch_key = key_batch[0]       # system
            categorical_covariates = None  # no donor
            
        result_sysvi = subprocess.run([
            "conda", "run", "-p", py_analysis, "python", "run_sysvi.py",
            str(n), adata_file, model_dir, emb_dir, 
            batch_key, ",".join(categorical_covariates) if categorical_covariates else "None",
            key_save, str(latent_dims), str(embedding_dims)], 
                                     capture_output=True, text=True)
        print(result_sysvi.stderr)
